# Crime Analytics & Prediction — Chicago Dataset

A machine learning pipeline for urban crime analysis built on the Chicago Crimes dataset (2001–Present). The project covers crime type classification, geographic hotspot detection, time-series forecasting, and text-based crime categorisation from incident descriptions.


## Table of Contents

| # | Section |
|---|---|
| 1 | Environment Setup |
| 2 | Dataset Loading |
| 3 | Data Cleaning |
| 4 | Feature Engineering |
| 5 | Exploratory Data Analysis |
| 6 | Classification — Random Forest vs XGBoost |
| 7 | Hyperparameter Tuning |
| 8 | NLP Classification (TF-IDF) |
| 9 | Hotspot Detection (Clustering) |
| 10 | Time-Series Forecasting (Prophet) |
| 11 | Model Comparison |
| 12 | Export Models |

Dataset: [Chicago Crimes — 2001 to Present](https://www.kaggle.com/datasets/currie32/crimes-in-chicago)  
GPU: Enable T4 in Kaggle → Settings → Accelerator

## Section 1 — Environment Setup

Kaggle ships with `scikit-learn`, `xgboost`, `pandas`, `numpy`, and `matplotlib` pre-installed. The only additional dependency is `prophet`, which needs to be installed manually.

In [ ]:
# Install prophet (not pre-installed in Kaggle)
# Run this cell first and wait for it to complete before continuing
!pip install prophet --quiet

In [ ]:
# SECTION 1 — IMPORT LIBRARIES

# --- Core Data Science ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# --- Scikit-learn: Preprocessing ---
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# --- Scikit-learn: Models ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans, DBSCAN

# --- Scikit-learn: Evaluation ---
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

# --- NLP ---
from sklearn.feature_extraction.text import TfidfVectorizer

# --- XGBoost ---
from xgboost import XGBClassifier, plot_importance

# --- Time Series ---
from prophet import Prophet

# --- Model Persistence ---
import joblib
import pickle

# --- Utility ---
import warnings
warnings.filterwarnings('ignore')

# --- Reproducibility ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Plot Style ---
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12

print("✅ All libraries imported successfully.")

## Section 2 — Dataset

The Chicago Crimes dataset has over 7 million records. We load 300,000 rows — enough to be statistically representative while staying within Kaggle's free-tier memory limits.

Columns used in this notebook:

| Column | Role |
|---|---|
| `Date` | Timestamp → temporal features |
| `Primary Type` | Classification target |
| `Description` | Free text → NLP model |
| `Latitude` / `Longitude` | Geographic features + clustering |
| `Arrest` | EDA only |


> After adding the dataset in Kaggle, verify the file path in the right panel and update the `DATASET_PATH` variable if needed.

In [ ]:
# SECTION 2 — LOAD DATASET

# Load 300,000 rows (full dataset is 7M+ rows — too large for free tier)
# Update the path to match your Kaggle dataset path
DATASET_PATH = '/kaggle/input/crimes-in-chicago/Chicago_Crimes_2001_to_2004.csv'

df = pd.read_csv(
    DATASET_PATH,
    nrows=300_000,
    low_memory=False
)

print(f"Dataset Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()

In [ ]:
# SECTION 2B — DATA OVERVIEW

print("=" * 60)
print("DATASET INFO")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("MISSING VALUES PER COLUMN")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_report[missing_report['Missing Count'] > 0])

print("\n" + "=" * 60)
print("BASIC STATISTICS")
print("=" * 60)
df.describe()

## Section 3 — Data Cleaning

Four cleaning steps are applied in order:

1. Parse the `Date` column into proper `datetime` — the raw format is `MM/DD/YYYY HH:MM:SS AM/PM`
2. Drop rows missing `Primary Type`, `Latitude`, `Longitude`, or `Description`
3. Remove duplicates on `Case Number`
4. Filter to Chicago's bounding box (Lat 41.6–42.1, Lon -87.9–-87.5) to remove geocoding errors

In [ ]:
# SECTION 3 — DATA CLEANING


initial_size = len(df)

# Step 1: Parse Date column
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')

# Step 2: Drop rows missing critical columns
critical_cols = ['Primary Type', 'Latitude', 'Longitude', 'Description']
df = df.dropna(subset=critical_cols)

# Step 3: Remove duplicates (by Case Number if available, else all columns)
if 'Case Number' in df.columns:
    df = df.drop_duplicates(subset=['Case Number'])
else:
    df = df.drop_duplicates()

# Step 4: Filter geographic outliers (Chicago bounding box)
df = df[
    (df['Latitude'].between(41.6, 42.1)) &
    (df['Longitude'].between(-87.9, -87.5))
]

# Step 5: Drop rows where Date parsing failed
df = df.dropna(subset=['Date'])

# Step 6: Reset index
df = df.reset_index(drop=True)

final_size = len(df)
print(f"Initial rows  : {initial_size:,}")
print(f"After cleaning: {final_size:,}")
print(f"Rows removed  : {initial_size - final_size:,} ({(initial_size - final_size)/initial_size*100:.1f}%)")
print(f"\n✅ Dataset is clean and ready for feature engineering.")

## Section 4 — Feature Engineering

All temporal features are derived from the `Date` column. The label encoder is saved here because it's needed by the Flask API later.

| Feature | Derived from | Notes |
|---|---|---|
| `Year` | Date | |
| `Month` | Date | |
| `Hour` | Date | |
| `DayOfWeek` | Date | 0 = Monday |
| `IsWeekend` | DayOfWeek | Binary |
| `Season` | Month | 0=Winter, 1=Spring, 2=Summer, 3=Fall |
| `crime_label` | Primary Type | Integer-encoded target |

In [ ]:
# SECTION 4 — FEATURE ENGINEERING

# Temporal features
df['Year']      = df['Date'].dt.year
df['Month']     = df['Date'].dt.month
df['Hour']      = df['Date'].dt.hour
df['DayOfWeek'] = df['Date'].dt.dayofweek   # 0=Monday, 6=Sunday

# Derived binary & categorical features
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
df['Season']    = df['Month'].apply(
    lambda m: 0 if m in [12,1,2] else 1 if m in [3,4,5] else 2 if m in [6,7,8] else 3
)   # 0=Winter, 1=Spring, 2=Summer, 3=Fall

# Target: encode Primary Type as integer labels
le = LabelEncoder()
df['crime_label'] = le.fit_transform(df['Primary Type'])

# Save the label encoder for later use in the Flask API
joblib.dump(le, 'label_encoder.pkl')

print(f"Number of unique crime types: {df['crime_label'].nunique()}")
print(f"\nCrime label mapping (first 10):")
for i, cls in enumerate(le.classes_[:10]):
    print(f"  {i:2d} → {cls}")

print(f"\n✅ Feature engineering complete. New shape: {df.shape}")
df[['Date','Year','Month','Hour','DayOfWeek','IsWeekend','Season','Primary Type','crime_label']].head()

## Section 5 — Exploratory Data Analysis

Four visualisations are produced here and saved as PNGs for the paper:

- Top 15 crime types by frequency
- Crime volume by year, month, hour, and day of week
- Geographic scatter and hex-bin density map
- Arrest rate by crime type

In [ ]:
# SECTION 5A — EDA: TOP CRIME TYPES

top_crimes = df['Primary Type'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_crimes.index[::-1], top_crimes.values[::-1],
               color=sns.color_palette('mako', 15))

# Add value labels
for bar, val in zip(bars, top_crimes.values[::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

ax.set_title('Top 15 Crime Types — Chicago Dataset', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Number of Incidents')
ax.set_ylabel('Crime Type')
sns.despine()
plt.tight_layout()
plt.savefig('eda_crime_types.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: eda_crime_types.png")

In [ ]:
# SECTION 5B — EDA: TEMPORAL PATTERNS

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Crime count by Year
df.groupby('Year').size().plot(ax=axes[0,0], marker='o', color='steelblue', linewidth=2)
axes[0,0].set_title('Crime Count by Year', fontweight='bold')
axes[0,0].set_xlabel('Year'); axes[0,0].set_ylabel('Count')

# 2. Crime count by Month
monthly = df.groupby('Month').size()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[0,1].bar(month_names, monthly.values, color=sns.color_palette('coolwarm', 12))
axes[0,1].set_title('Crime Count by Month (Seasonality)', fontweight='bold')
axes[0,1].set_xlabel('Month'); axes[0,1].set_ylabel('Count')

# 3. Crime count by Hour
hourly = df.groupby('Hour').size()
axes[1,0].plot(hourly.index, hourly.values, color='coral', linewidth=2, marker='o', markersize=4)
axes[1,0].fill_between(hourly.index, hourly.values, alpha=0.2, color='coral')
axes[1,0].set_title('Crime Count by Hour of Day', fontweight='bold')
axes[1,0].set_xlabel('Hour (0–23)'); axes[1,0].set_ylabel('Count')

# 4. Crime count by Day of Week
day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow = df.groupby('DayOfWeek').size()
colors = ['#d9534f' if i >= 5 else '#5bc0de' for i in range(7)]  # Red for weekends
axes[1,1].bar(day_names, dow.values, color=colors)
axes[1,1].set_title('Crime Count by Day of Week\n(Red = Weekend)', fontweight='bold')
axes[1,1].set_xlabel('Day'); axes[1,1].set_ylabel('Count')

plt.suptitle('Temporal Crime Patterns — Chicago Dataset', fontsize=16, fontweight='bold', y=1.01)
sns.despine()
plt.tight_layout()
plt.savefig('eda_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: eda_temporal.png")

In [ ]:
# SECTION 5C — EDA: GEOGRAPHIC DISTRIBUTION

# Sample 50k points for cleaner scatter (plotting all 300k is very slow)
sample = df.sample(n=min(50_000, len(df)), random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Scatter plot
axes[0].scatter(
    sample['Longitude'], sample['Latitude'],
    alpha=0.08, s=1, c='crimson'
)
axes[0].set_title('Crime Location Scatter Plot\n(50k sample)', fontweight='bold')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# 2D Density (Hex bin) — shows hotspot density
hb = axes[1].hexbin(
    sample['Longitude'], sample['Latitude'],
    gridsize=50, cmap='YlOrRd', mincnt=1
)
plt.colorbar(hb, ax=axes[1], label='Crime Count')
axes[1].set_title('Crime Density Heatmap (Hex Bin)\n(50k sample)', fontweight='bold')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

plt.suptitle('Geographic Distribution of Crimes — Chicago', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_geographic.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: eda_geographic.png")

In [ ]:
# SECTION 5D — EDA: ARREST RATE BY CRIME TYPE

# Only run if Arrest column exists
if 'Arrest' in df.columns:
    arrest_rate = (
        df.groupby('Primary Type')['Arrest']
        .mean()
        .sort_values(ascending=False)
        .head(15)
        * 100
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(
        arrest_rate.index[::-1],
        arrest_rate.values[::-1],
        color=sns.color_palette('RdYlGn', 15)
    )
    ax.set_title('Arrest Rate (%) by Crime Type — Top 15', fontsize=13, fontweight='bold')
    ax.set_xlabel('Arrest Rate (%)')
    ax.axvline(x=arrest_rate.mean(), color='navy', linestyle='--', label=f'Mean: {arrest_rate.mean():.1f}%')
    ax.legend()
    sns.despine()
    plt.tight_layout()
    plt.savefig('eda_arrest_rate.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Figure saved: eda_arrest_rate.png")
else:
    print("⚠️ 'Arrest' column not found in dataset — skipping this plot.")

## Section 6 — Classification: Random Forest vs XGBoost

The task is predicting `Primary Type` from the seven spatio-temporal features above. Two models are trained and compared.

Features: `Latitude`, `Longitude`, `Hour`, `Month`, `DayOfWeek`, `IsWeekend`, `Season`

**Random Forest** is trained first as a baseline. `class_weight='balanced'` is set because the dataset is heavily skewed toward THEFT — without it the model just learns to predict the majority class.

**XGBoost** generally outperforms RF on tabular data of this kind. `tree_method='hist'` is used for faster training; change `device='cpu'` to `device='cuda'` if a GPU is attached.

> A note on accuracy: 25–45% on a 30-class problem where many classes share the same times and locations is expected. This is a hard multi-class problem, not a binary one. The NLP model in Section 8 will score higher because description text is much more discriminative than coordinates alone.

In [ ]:
# SECTION 6A — PREPARE FEATURES FOR CLASSIFICATION

FEATURES = ['Latitude', 'Longitude', 'Hour', 'Month', 'DayOfWeek', 'IsWeekend', 'Season']
TARGET   = 'crime_label'

X = df[FEATURES].copy()
y = df[TARGET].copy()

# Train/Test split — 80/20, stratified to preserve class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set : {X_train.shape[0]:,} samples")
print(f"Test set     : {X_test.shape[0]:,} samples")
print(f"Features     : {FEATURES}")
print(f"Target classes: {y.nunique()}")

In [ ]:
# SECTION 6B — MODEL 1: RANDOM FOREST CLASSIFIER


print("Training Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    class_weight='balanced',   # handles class imbalance
    random_state=RANDOM_STATE,
    n_jobs=-1                  # use all CPU cores
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
print(f"\n✅ Random Forest Accuracy: {rf_acc:.4f} ({rf_acc*100:.2f}%)")
print("\nDetailed Classification Report:")
print(classification_report(y_test, rf_pred, target_names=le.classes_[:10], labels=list(range(10))))

In [ ]:
# SECTION 6C — MODEL 2: XGBOOST CLASSIFIER

print("Training XGBoost...")

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    tree_method='hist',        # fast training — works on CPU and GPU
    device='cpu'               # change to 'cuda' if GPU is enabled
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

xgb_pred = xgb_model.predict(X_test)

xgb_acc = accuracy_score(y_test, xgb_pred)
print(f"\n✅ XGBoost Accuracy: {xgb_acc:.4f} ({xgb_acc*100:.2f}%)")
print("\nDetailed Classification Report:")
print(classification_report(y_test, xgb_pred, target_names=le.classes_[:10], labels=list(range(10))))

In [ ]:
# SECTION 6D — CONFUSION MATRICES (TOP 10 CLASSES)

# Filter to top 10 most common classes for readability
top10_labels = list(range(10))
top10_names  = le.classes_[:10].tolist()

# Filter predictions to top 10 only
mask = y_test.isin(top10_labels)
y_test_top10   = y_test[mask]
rf_pred_top10  = rf_pred[mask]
xgb_pred_top10 = xgb_pred[mask]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, pred, title in zip(axes,
                            [rf_pred_top10, xgb_pred_top10],
                            ['Random Forest', 'XGBoost']):
    cm = confusion_matrix(y_test_top10, pred, labels=top10_labels)
    sns.heatmap(
        cm, ax=ax,
        xticklabels=top10_names,
        yticklabels=top10_names,
        cmap='Blues', fmt='d', annot=True, annot_kws={'size': 7}
    )
    ax.set_title(f'Confusion Matrix — {title}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', rotation=0, labelsize=7)

plt.suptitle('Confusion Matrices — Top 10 Crime Classes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: confusion_matrices.png")

In [ ]:
# SECTION 6E — FEATURE IMPORTANCE COMPARISON

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest feature importance
rf_importances = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values()
rf_importances.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Feature Importance — Random Forest', fontweight='bold')
axes[0].set_xlabel('Importance Score')

# XGBoost feature importance
xgb_importances = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values()
xgb_importances.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Feature Importance — XGBoost', fontweight='bold')
axes[1].set_xlabel('Importance Score')

sns.despine()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: feature_importance.png")

## Section 7 — Hyperparameter Tuning

Grid search with 3-fold cross-validation over `max_depth`, `learning_rate`, `n_estimators`, and `subsample`. To keep runtime under 10 minutes on a free Kaggle session, tuning runs on a stratified 20% subsample of the training set.

In [ ]:
# SECTION 7 — HYPERPARAMETER TUNING (GRID SEARCH)

# Use 20% subsample for tuning speed (still statistically valid)
X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train,
    test_size=0.80,
    random_state=RANDOM_STATE,
    stratify=y_train
)

print(f"Tuning on {X_tune.shape[0]:,} samples...")

param_grid = {
    'max_depth'    : [6, 8],
    'learning_rate': [0.05, 0.1],
    'n_estimators' : [100, 200],
    'subsample'    : [0.8]
}

base_xgb = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    tree_method='hist'
)

grid_search = GridSearchCV(
    base_xgb,
    param_grid,
    cv=3,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_tune, y_tune)

print(f"\n✅ Best Parameters: {grid_search.best_params_}")
print(f"   Best CV Accuracy: {grid_search.best_score_:.4f}")

# Evaluate best model on held-out test set
best_xgb = grid_search.best_estimator_
best_pred = best_xgb.predict(X_test)
best_acc  = accuracy_score(y_test, best_pred)
print(f"   Test Set Accuracy (best model): {best_acc:.4f}")

## Section 8 — NLP Classification (TF-IDF + Logistic Regression)

Incident descriptions like "RETAIL THEFT OVER $500" contain information that coordinates alone cannot capture. This section builds a text classification pipeline:

- TF-IDF with unigrams and bigrams, max 10,000 features, log-normalised (`sublinear_tf=True`)
- Logistic Regression with `multinomial` solver

The pipeline is scoped to the top 10 crime classes to reduce label noise. A demo function at the end of the section lets you pass any description string and see the predicted class with confidence.

In [ ]:
# SECTION 8 — NLP CRIME CLASSIFICATION (TF-IDF)

# Ensure Description is string type and lowercase
df['Description'] = df['Description'].astype(str).str.lower().str.strip()

# Filter to top 10 crime classes to reduce complexity
top10_types = df['Primary Type'].value_counts().head(10).index.tolist()
df_nlp = df[df['Primary Type'].isin(top10_types)].copy()

# Re-encode labels for this filtered subset
le_nlp = LabelEncoder()
df_nlp['nlp_label'] = le_nlp.fit_transform(df_nlp['Primary Type'])
joblib.dump(le_nlp, 'nlp_label_encoder.pkl')

X_text = df_nlp['Description']
y_text = df_nlp['nlp_label']

# Train/Test split
X_txt_train, X_txt_test, y_txt_train, y_txt_test = train_test_split(
    X_text, y_text,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_text
)

# TF-IDF Vectorization
print("Fitting TF-IDF vectorizer...")
vectorizer = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),        # unigrams + bigrams
    min_df=3,                  # ignore very rare terms
    sublinear_tf=True          # apply log normalization
)

X_tfidf_train = vectorizer.fit_transform(X_txt_train)
X_tfidf_test  = vectorizer.transform(X_txt_test)

print(f"TF-IDF Matrix Shape (train): {X_tfidf_train.shape}")

# Logistic Regression NLP Model
print("\nTraining NLP classifier (Logistic Regression)...")
nlp_model = LogisticRegression(
    max_iter=500,
    C=1.0,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    solver='lbfgs',
    multi_class='multinomial',
    n_jobs=-1
)

nlp_model.fit(X_tfidf_train, y_txt_train)

nlp_pred = nlp_model.predict(X_tfidf_test)
nlp_acc  = accuracy_score(y_txt_test, nlp_pred)

print(f"\n✅ NLP Model Accuracy: {nlp_acc:.4f} ({nlp_acc*100:.2f}%)")
print("\nDetailed Report:")
print(classification_report(y_txt_test, nlp_pred, target_names=le_nlp.classes_))

In [ ]:
# SECTION 8B — NLP DEMO: PREDICT A DESCRIPTION

def predict_crime_from_description(description_text):
    """Predict crime type from a free-text description."""
    vec = vectorizer.transform([description_text.lower()])
    pred_label = nlp_model.predict(vec)[0]
    pred_proba = nlp_model.predict_proba(vec)[0]
    pred_class = le_nlp.inverse_transform([pred_label])[0]
    confidence = pred_proba.max() * 100
    return pred_class, confidence

# Example predictions
test_descriptions = [
    "retail theft from store, shoplifting merchandise",
    "aggravated assault with knife on public street",
    "vehicle break-in, smashed window and stole items",
    "possession of controlled substance, narcotics found"
]

print("NLP Model Prediction Demo:")
print("-" * 55)
for desc in test_descriptions:
    cls, conf = predict_crime_from_description(desc)
    print(f"Input : {desc[:50]}...")
    print(f"Output: {cls} (confidence: {conf:.1f}%)")
    print("-" * 55)

## Section 9 — Hotspot Detection

Two clustering approaches are compared on geographic coordinates.

**K-Means** requires specifying K upfront. An elbow curve is plotted first to justify the choice of K=12.

**DBSCAN** finds clusters based on density without a fixed K, and marks sparse points as noise. It's run with `eps=0.005` (roughly 500m at Chicago's latitude) using the haversine metric on radians. This tends to produce more geographically coherent clusters than K-Means for crime data, where hotspot shapes are irregular.

In [ ]:
# SECTION 9A — K-MEANS HOTSPOT DETECTION

# Use a sample for speed; clustering all 300k rows can be slow
geo_sample = df[['Latitude', 'Longitude']].sample(n=min(80_000, len(df)), random_state=RANDOM_STATE)

# Elbow method to find optimal K
print("Running elbow method (K=2 to 20)...")
inertias = []
K_range = range(2, 21)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=5, max_iter=100)
    km.fit(geo_sample)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(K_range, inertias, marker='o', color='steelblue', linewidth=2)
ax.set_title('Elbow Method — Optimal Number of Clusters (K)', fontweight='bold')
ax.set_xlabel('K (Number of Clusters)')
ax.set_ylabel('Inertia (Within-Cluster Sum of Squares)')
sns.despine()
plt.tight_layout()
plt.savefig('elbow_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: elbow_curve.png")

In [ ]:
# SECTION 9B — FIT K-MEANS WITH OPTIMAL K

OPTIMAL_K = 12   # Adjust based on elbow curve output above

kmeans_model = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
geo_sample = geo_sample.copy()
geo_sample['kmeans_cluster'] = kmeans_model.fit_predict(geo_sample)

# Also assign clusters to the full dataframe for analysis
df['kmeans_cluster'] = kmeans_model.predict(df[['Latitude', 'Longitude']])

# Plot clusters
fig, ax = plt.subplots(figsize=(10, 9))
scatter = ax.scatter(
    geo_sample['Longitude'],
    geo_sample['Latitude'],
    c=geo_sample['kmeans_cluster'],
    cmap='tab20',
    s=1,
    alpha=0.4
)

# Plot cluster centers
centers = kmeans_model.cluster_centers_
ax.scatter(centers[:, 1], centers[:, 0], c='black', s=100, marker='X', zorder=5, label='Cluster Centers')
ax.set_title(f'K-Means Crime Hotspots (K={OPTIMAL_K}) — Chicago', fontsize=13, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend()
plt.colorbar(scatter, ax=ax, label='Cluster ID')
plt.tight_layout()
plt.savefig('kmeans_hotspots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: kmeans_hotspots.png")

In [ ]:
# SECTION 9C — DBSCAN DENSITY-BASED CLUSTERING

# DBSCAN on a smaller sample (it's O(n^2) in worst case)
dbscan_sample = df[['Latitude', 'Longitude']].sample(n=min(30_000, len(df)), random_state=RANDOM_STATE).copy()

# eps in degrees (~0.005° ≈ 500m in Chicago)
dbscan_model = DBSCAN(eps=0.005, min_samples=50, algorithm='ball_tree', metric='haversine')
dbscan_sample['dbscan_cluster'] = dbscan_model.fit_predict(
    np.radians(dbscan_sample[['Latitude', 'Longitude']])
)

n_clusters_found = len(set(dbscan_sample['dbscan_cluster'])) - (1 if -1 in dbscan_sample['dbscan_cluster'].values else 0)
n_noise          = (dbscan_sample['dbscan_cluster'] == -1).sum()

print(f"DBSCAN Results:")
print(f"  Clusters found : {n_clusters_found}")
print(f"  Noise points   : {n_noise:,} ({n_noise/len(dbscan_sample)*100:.1f}%)")

# Plot
fig, ax = plt.subplots(figsize=(10, 9))

# Noise points in grey
noise = dbscan_sample[dbscan_sample['dbscan_cluster'] == -1]
clusters = dbscan_sample[dbscan_sample['dbscan_cluster'] != -1]

ax.scatter(noise['Longitude'], noise['Latitude'], c='lightgrey', s=1, alpha=0.3, label='Noise')
scatter = ax.scatter(
    clusters['Longitude'], clusters['Latitude'],
    c=clusters['dbscan_cluster'], cmap='tab20', s=2, alpha=0.6
)
ax.set_title(f'DBSCAN Crime Hotspots ({n_clusters_found} clusters) — Chicago', fontsize=13, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend(markerscale=4)
plt.tight_layout()
plt.savefig('dbscan_hotspots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: dbscan_hotspots.png")

## Section 10 — Time-Series Forecasting (Prophet)

Daily crime counts are aggregated from the cleaned dataset and passed to Prophet. The model is configured with multiplicative seasonality (crime volume scales with the trend, so multiplicative fits better than additive), yearly and weekly components enabled, and US holidays included.

The forecast plot and components decomposition (trend + weekly + yearly seasonality) are both saved as PNGs.

In [ ]:
# SECTION 10A — PREPARE TIME SERIES DATA

# Aggregate to daily crime counts
# Prophet requires columns named 'ds' (date) and 'y' (value)
crime_by_day = (
    df.groupby(df['Date'].dt.date)
    .size()
    .reset_index()
)
crime_by_day.columns = ['ds', 'y']
crime_by_day['ds'] = pd.to_datetime(crime_by_day['ds'])

# Sort chronologically
crime_by_day = crime_by_day.sort_values('ds').reset_index(drop=True)

print(f"Time series range: {crime_by_day['ds'].min().date()} → {crime_by_day['ds'].max().date()}")
print(f"Total days: {len(crime_by_day)}")
print(f"Avg crimes/day: {crime_by_day['y'].mean():.1f}")

# Plot raw time series
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(crime_by_day['ds'], crime_by_day['y'], color='steelblue', linewidth=0.8, alpha=0.8)
ax.set_title('Daily Crime Count Time Series', fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Crime Count')
ax.fill_between(crime_by_day['ds'], crime_by_day['y'], alpha=0.2, color='steelblue')
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# SECTION 10B — TRAIN PROPHET FORECASTING MODEL

print("Training Prophet model...")

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative',    # better for crime data (scales with volume)
    changepoint_prior_scale=0.1,          # controls trend flexibility
    uncertainty_samples=1000
)

# Add US holidays (affects crime patterns)
prophet_model.add_country_holidays(country_name='US')

prophet_model.fit(crime_by_day)

print("✅ Prophet model trained successfully.")

In [ ]:
# SECTION 10C — GENERATE 30-DAY FORECAST

FORECAST_DAYS = 30

future = prophet_model.make_future_dataframe(periods=FORECAST_DAYS, freq='D')
forecast = prophet_model.predict(future)

# Main forecast plot
fig = prophet_model.plot(forecast, figsize=(14, 5))
plt.title(f'Crime Forecast — Next {FORECAST_DAYS} Days (Prophet)', fontsize=13, fontweight='bold')
plt.xlabel('Date'); plt.ylabel('Daily Crime Count')
plt.tight_layout()
plt.savefig('prophet_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

# Components plot (trend + seasonality decomposition)
fig2 = prophet_model.plot_components(forecast, figsize=(12, 8))
plt.suptitle('Prophet Forecast Components (Trend + Seasonality)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('prophet_components.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Figures saved: prophet_forecast.png, prophet_components.png")

# Show upcoming 30-day forecast
upcoming = forecast[forecast['ds'] > crime_by_day['ds'].max()][['ds','yhat','yhat_lower','yhat_upper']].copy()
upcoming.columns = ['Date', 'Predicted Crimes', 'Lower Bound', 'Upper Bound']
upcoming['Date'] = upcoming['Date'].dt.date
upcoming[['Predicted Crimes','Lower Bound','Upper Bound']] = upcoming[['Predicted Crimes','Lower Bound','Upper Bound']].round(1)
print("\n30-Day Crime Forecast:")
upcoming.head(10)

## Section 11 — Model Comparison & Evaluation Summary

Below is a consolidated summary of all models trained in this notebook. This table is what you would include in the **Results** section of a research paper.

In [ ]:
# SECTION 11 — MODEL EVALUATION SUMMARY TABLE

from sklearn.metrics import f1_score, precision_score, recall_score

# Compute metrics for all classifiers
def get_metrics(y_true, y_pred, model_name):
    return {
        'Model'    : model_name,
        'Accuracy' : round(accuracy_score(y_true, y_pred) * 100, 2),
        'Precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0) * 100, 2),
        'Recall'   : round(recall_score(y_true, y_pred, average='weighted', zero_division=0) * 100, 2),
        'F1-Score' : round(f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100, 2),
    }

results = [
    get_metrics(y_test, rf_pred,   'Random Forest (Spatial)'),
    get_metrics(y_test, xgb_pred,  'XGBoost (Spatial)'),
    get_metrics(y_test, best_pred, 'XGBoost Tuned (Spatial)'),
    get_metrics(y_txt_test, nlp_pred, 'Logistic Regression (NLP/TF-IDF)'),
]

results_df = pd.DataFrame(results).set_index('Model')

print("\n" + "=" * 65)
print("         MODEL COMPARISON — CLASSIFICATION RESULTS")
print("=" * 65)
print(results_df.to_string())
print("=" * 65)

# Visualise comparison
ax = results_df.plot(
    kind='bar', figsize=(13, 5),
    colormap='Set2', edgecolor='black', linewidth=0.5
)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 110)
ax.tick_params(axis='x', rotation=20)
ax.legend(loc='upper right')

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', fontsize=8, padding=2)

sns.despine()
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: model_comparison.png")

## Section 12 — Save Models for Deployment

We save all trained models to disk. These will be loaded by the **Flask API** for real-time predictions in the deployment phase.

| File | Contents | Used by |
|---|---|---|
| `xgb_crime_model.pkl` | Best XGBoost classifier | Flask `/predict` endpoint |
| `rf_crime_model.pkl` | Random Forest classifier | Flask `/predict` endpoint |
| `nlp_model.pkl` | Logistic Regression NLP model | Flask `/nlp-classify` endpoint |
| `tfidf_vectorizer.pkl` | TF-IDF vectorizer | Flask `/nlp-classify` endpoint |
| `prophet_model.pkl` | Prophet forecasting model | Flask `/forecast` endpoint |
| `label_encoder.pkl` | Crime type label encoder | All classification endpoints |
| `nlp_label_encoder.pkl` | NLP crime type encoder | NLP endpoint |
| `kmeans_model.pkl` | K-Means clustering model | Flask `/hotspot` endpoint |

In [ ]:
# SECTION 12 — SAVE ALL MODELS

import os

# Create output directory
os.makedirs('/kaggle/working/models', exist_ok=True)

models_to_save = {
    '/kaggle/working/models/xgb_crime_model.pkl'    : best_xgb,
    '/kaggle/working/models/rf_crime_model.pkl'     : rf_model,
    '/kaggle/working/models/nlp_model.pkl'          : nlp_model,
    '/kaggle/working/models/tfidf_vectorizer.pkl'   : vectorizer,
    '/kaggle/working/models/prophet_model.pkl'      : prophet_model,
    '/kaggle/working/models/label_encoder.pkl'      : le,
    '/kaggle/working/models/nlp_label_encoder.pkl'  : le_nlp,
    '/kaggle/working/models/kmeans_model.pkl'       : kmeans_model,
}

print("Saving models...")
for path, model_obj in models_to_save.items():
    joblib.dump(model_obj, path)
    size_kb = os.path.getsize(path) / 1024
    print(f"  ✅ Saved: {os.path.basename(path):40s} ({size_kb:.1f} KB)")

print("\n🚀 All models saved successfully to /kaggle/working/models/")
print("\nNext Steps:")
print("  1. Download these .pkl files from the Kaggle output panel")
print("  2. Place them in your Flask backend /models/ directory")
print("  3. Build Flask API to serve these models (Phase 2)")
print("  4. Build React dashboard (Phase 3)")

## Conclusion

XGBoost on spatio-temporal features and Logistic Regression on TF-IDF descriptions are the two strongest classifiers. DBSCAN produces more useful hotspot boundaries than K-Means because Chicago crime clusters are not spherical. Prophet captures the strong weekly seasonality in the data and gives reasonable 30-day forecasts with uncertainty intervals.

**Planned extensions:**
- Fine-tune DistilBERT on the description field to replace the TF-IDF baseline
- Replace Prophet with an LSTM for the forecasting component
- Add socioeconomic and weather covariates as Prophet regressors
- Build an interactive Folium map integrated with the React dashboard


**Author:** Muhammad Aqib Javed  
**Institution:** UET Taxila  
**Dataset:** Chicago Crimes (Kaggle)  
**Stack:** Python 3.10 · scikit-learn · XGBoost · Prophet · Pandas · Seaborn